# SSVI surface diagnostics

Fit SSVI globally and Raw SVI independently by expiry on the screened 2025-08-29 SPX quotes. This notebook compares their fitted total-variance slices.

Saved quote-bearing outputs are intentionally omitted from the public notebook. Run locally with licensed data; see [notebook guide](README.md). This is an exploratory comparator; production uses Raw SVI plus calendar repair. Manual slider interaction is optional.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))
from src.raw_svi import fit_raw_svi_surface, raw_svi_total_variance
from src.ssvi import fit_ssvi_surface, ssvi_total_variance

PANEL_PATH = Path("../data/processed/spx_iv_panel_2025-08-29.csv")
panel = pd.read_csv(PANEL_PATH, parse_dates=["quote_date", "expiry_date"])
result = fit_ssvi_surface(panel)
raw_result = fit_raw_svi_surface(panel)
observations = result["observations"]
raw_observations = raw_result["observations"]

display(pd.Series({key: result[key] for key in ["rho", "eta", "gamma", "rmse"]}))
display(result["theta_by_expiry"])
display(raw_result["parameters_by_expiry"])


## Representative expiries

For each target DTE, select the closest available expiry. Compare the shared-parameter SSVI fit with a separate five-parameter Raw SVI fit for that expiry. Both are fitted to the same screened OTM midpoint total-variance observations. Some Raw SVI fits reach parameter bounds, so treat their low in-sample errors and parameters cautiously.

In [ ]:
expiry_table = observations[["expiry_date", "days_to_expiry"]].drop_duplicates()
targets = [7, 30, 90, 300]
selected_expiries = []
for target in targets:
    nearest = expiry_table.iloc[(expiry_table["days_to_expiry"] - target).abs().argmin()]
    if nearest["expiry_date"] not in selected_expiries:
        selected_expiries.append(nearest["expiry_date"])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, expiry in zip(axes.flat, selected_expiries):
    sample = observations.loc[observations["expiry_date"].eq(expiry)]
    k_grid = np.linspace(sample["log_moneyness"].min(), sample["log_moneyness"].max(), 300)
    theta = sample["theta_atm"].iloc[0]
    ax.scatter(sample["log_moneyness"], sample["market_w"], s=12, alpha=0.65, label="Observed")
    ax.plot(
        k_grid,
        ssvi_total_variance(k_grid, theta, result["rho"], result["eta"], result["gamma"]),
        color="tab:red", label="Fitted SSVI",
    )
    raw_parameters = raw_result["parameters_by_expiry"].loc[
        raw_result["parameters_by_expiry"]["expiry_date"].eq(expiry)
    ].iloc[0]
    raw_params = [raw_parameters[name] for name in ["a", "b", "rho", "m", "sigma"]]
    ax.plot(
        k_grid, raw_svi_total_variance(k_grid, *raw_params),
        color="tab:green", linestyle="--", label="Fitted Raw SVI",
    )
    ax.set(title=f"{expiry.date()} ({sample['days_to_expiry'].iloc[0]} DTE)", xlabel="Log-moneyness", ylabel="Total variance")
    ax.legend()
    ax.grid(alpha=0.25)
for ax in axes.flat[len(selected_expiries):]:
    ax.set_visible(False)
fig.tight_layout()
plt.show()

## Residuals

The residual is fitted minus observed total variance. Inspect whether errors change systematically with log-moneyness or maturity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(observations["log_moneyness"], observations["residual_w"], s=6, alpha=0.4)
axes[1].scatter(observations["days_to_expiry"], observations["residual_w"], s=6, alpha=0.4)
for ax, xlabel in zip(axes, ["Log-moneyness", "Days to expiry"]):
    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set(xlabel=xlabel, ylabel="Fitted − observed total variance")
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## Observed and fitted IV in 3D

The left plot shows the filtered observed IVs. The right plot shows fitted SSVI curves for every expiry, each over the log-moneyness range of that expiry's quotes. The curves are separate expiry slices; no interpolation between expiries is fitted.

In [ ]:
fig = plt.figure(figsize=(15, 6), layout="constrained")
observed_ax = fig.add_subplot(121, projection="3d")
fitted_ax = fig.add_subplot(122, projection="3d")

points = observed_ax.scatter(
    observations["log_moneyness"],
    observations["days_to_expiry"],
    observations["mid_iv"],
    c=observations["mid_iv"], s=3, alpha=0.5, cmap="viridis",
)
fig.colorbar(points, ax=observed_ax, shrink=0.6, pad=0.1, label="Observed IV")

for expiry, sample in observations.groupby("expiry_date"):
    k_grid = np.linspace(sample["log_moneyness"].min(), sample["log_moneyness"].max(), 100)
    theta = sample["theta_atm"].iloc[0]
    tau = sample["time_to_expiry"].iloc[0]
    fitted_iv = np.sqrt(
        ssvi_total_variance(k_grid, theta, result["rho"], result["eta"], result["gamma"]) / tau
    )
    fitted_ax.plot(
        k_grid, np.full_like(k_grid, sample["days_to_expiry"].iloc[0]), fitted_iv,
        color="tab:blue", linewidth=1, alpha=0.7,
    )

for ax, title in [(observed_ax, "Observed midpoint IV"), (fitted_ax, "Fitted SSVI slices")]:
    ax.set(xlabel="Log-moneyness", ylabel="Days to expiry", zlabel="IV", title=title)
    ax.set_zlim(0, observations["mid_iv"].max())
    ax.view_init(elev=25, azim=-65)
plt.show()

## Rendered SSVI surface

Drag the Plotly figure to rotate it and scroll to zoom. The mesh connects fitted expiry slices where adjacent expiries both have filtered quotes. Each row uses that expiry's estimated $\theta_{\mathrm{ATM}}$ and the shared $\rho$, $\eta$, and $\gamma$. The facets between rows are a plotting connection, not an estimated continuous ATM-variance curve.

In [ ]:
expiry_ranges = observations.groupby("expiry_date").agg(
    days_to_expiry=("days_to_expiry", "first"),
    time_to_expiry=("time_to_expiry", "first"),
    theta_atm=("theta_atm", "first"),
    k_min=("log_moneyness", "min"),
    k_max=("log_moneyness", "max"),
).sort_values("days_to_expiry")

k_grid = np.linspace(observations["log_moneyness"].min(), observations["log_moneyness"].max(), 160)
k_mesh, dte_mesh = np.meshgrid(k_grid, expiry_ranges["days_to_expiry"])
theta = expiry_ranges["theta_atm"].to_numpy()[:, None]
tau = expiry_ranges["time_to_expiry"].to_numpy()[:, None]
iv_mesh = np.sqrt(
    ssvi_total_variance(k_mesh, theta, result["rho"], result["eta"], result["gamma"]) / tau
)
within_quotes = (
    (k_mesh >= expiry_ranges["k_min"].to_numpy()[:, None])
    & (k_mesh <= expiry_ranges["k_max"].to_numpy()[:, None])
)
iv_mesh = np.ma.masked_where(~within_quotes, iv_mesh)

import plotly.graph_objects as go

surface_fig = go.Figure(go.Surface(
    x=k_grid,
    y=expiry_ranges["days_to_expiry"].to_numpy(),
    z=iv_mesh.filled(np.nan),
    colorscale="Viridis",
    connectgaps=False,
    colorbar=dict(title="Fitted IV"),
    hovertemplate="k=%{x:.3f}<br>DTE=%{y}<br>IV=%{z:.3f}<extra></extra>",
))
surface_fig.update_layout(
    title="SSVI surface over quoted ranges",
    scene=dict(
        xaxis_title="Log-moneyness",
        yaxis_title="Days to expiry",
        zaxis_title="IV",
    ),
    width=900, height=650,
)
surface_fig.show()

## What changes without the quote screen?

This comparison uses every valid OTM midpoint IV (`use_for_surface`) from the existing panel, including zero-bid and wide-spread quotes. The panel's invalid or missing IVs still cannot be fitted. Each RMSE is calculated on its own sample, so compare the fitted parameters and shapes as well as the error.

In [ ]:
unscreened_result = fit_ssvi_surface(panel, apply_quote_screen=False)
unscreened_rows = unscreened_result["observations"]

comparison = pd.DataFrame({
    name: {
        "quotes": len(fit["observations"]),
        "rho": fit["rho"],
        "eta": fit["eta"],
        "gamma": fit["gamma"],
        "rmse": fit["rmse"],
    }
    for name, fit in [("screened", result), ("unscreened", unscreened_result)]
}).T
display(comparison)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, expiry in zip(axes.flat, selected_expiries):
    raw = unscreened_rows.loc[unscreened_rows["expiry_date"].eq(expiry)]
    clean = observations.loc[observations["expiry_date"].eq(expiry)]
    raw_k = np.linspace(raw["log_moneyness"].min(), raw["log_moneyness"].max(), 200)
    clean_k = np.linspace(clean["log_moneyness"].min(), clean["log_moneyness"].max(), 200)
    ax.scatter(raw["log_moneyness"], raw["market_w"], s=10, alpha=0.4, color="gray", label="All valid OTM IVs")
    ax.plot(raw_k, ssvi_total_variance(raw_k, raw["theta_atm"].iloc[0], unscreened_result["rho"], unscreened_result["eta"], unscreened_result["gamma"]), color="tab:orange", label="Unscreened fit")
    ax.plot(clean_k, ssvi_total_variance(clean_k, clean["theta_atm"].iloc[0], result["rho"], result["eta"], result["gamma"]), color="tab:blue", label="Screened fit")
    ax.set(title=f"{expiry.date()} ({raw['days_to_expiry'].iloc[0]} DTE)", xlabel="Log-moneyness", ylabel="Total variance")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
for ax in axes.flat[len(selected_expiries):]:
    ax.set_visible(False)
fig.tight_layout()
plt.show()

## Unscreened IVs and fitted surface

Drag to rotate. Black circles are observed midpoint IVs; black diamonds are SSVI values fitted at those exact quote locations. The coloured mesh connects the fitted expiry slices. All valid OTM observations enter the least-squares fit, but individual points need not lie on the surface. Hover over a circle to compare its observed and fitted IV.

In [ ]:
raw_ranges = unscreened_rows.groupby("expiry_date").agg(
    days_to_expiry=("days_to_expiry", "first"),
    time_to_expiry=("time_to_expiry", "first"),
    theta_atm=("theta_atm", "first"),
    k_min=("log_moneyness", "min"),
    k_max=("log_moneyness", "max"),
).sort_values("days_to_expiry")
raw_k_grid = np.linspace(unscreened_rows["log_moneyness"].min(), unscreened_rows["log_moneyness"].max(), 160)
raw_k_mesh, raw_dte_mesh = np.meshgrid(raw_k_grid, raw_ranges["days_to_expiry"])
raw_iv_mesh = np.sqrt(
    ssvi_total_variance(
        raw_k_mesh, raw_ranges["theta_atm"].to_numpy()[:, None],
        unscreened_result["rho"], unscreened_result["eta"], unscreened_result["gamma"],
    ) / raw_ranges["time_to_expiry"].to_numpy()[:, None]
)
raw_coverage = (
    (raw_k_mesh >= raw_ranges["k_min"].to_numpy()[:, None])
    & (raw_k_mesh <= raw_ranges["k_max"].to_numpy()[:, None])
)
raw_iv_mesh = np.where(raw_coverage, raw_iv_mesh, np.nan)

iv_error = unscreened_rows["mid_iv"] - unscreened_rows["fitted_iv"]
display(pd.Series({
    "observations_fitted": len(unscreened_rows),
    "missing_fitted_iv": unscreened_rows["fitted_iv"].isna().sum(),
    "median_absolute_iv_error": iv_error.abs().median(),
    "maximum_absolute_iv_error": iv_error.abs().max(),
}))

unscreened_fig = go.Figure()
unscreened_fig.add_trace(go.Surface(
    x=raw_k_grid, y=raw_ranges["days_to_expiry"].to_numpy(), z=raw_iv_mesh,
    colorscale="Viridis", opacity=0.55, connectgaps=False,
    colorbar=dict(title="Fitted IV"), name="Unscreened SSVI",
))
unscreened_fig.add_trace(go.Scatter3d(
    x=unscreened_rows["log_moneyness"],
    y=unscreened_rows["days_to_expiry"],
    z=unscreened_rows["mid_iv"],
    mode="markers", marker=dict(size=1.5, color="black", opacity=0.35),
    customdata=np.column_stack((unscreened_rows["fitted_iv"], iv_error)),
    hovertemplate="k=%{x:.3f}<br>DTE=%{y}<br>Observed IV=%{z:.3f}<br>Fitted IV=%{customdata[0]:.3f}<br>Error=%{customdata[1]:+.3f}<extra></extra>",
    name="Observed IV at each quote",
))
unscreened_fig.add_trace(go.Scatter3d(
    x=unscreened_rows["log_moneyness"],
    y=unscreened_rows["days_to_expiry"],
    z=unscreened_rows["fitted_iv"],
    mode="markers", marker=dict(size=1.5, color="black", symbol="diamond", opacity=0.6),
    name="Fitted IV at each quote",
))
unscreened_fig.update_layout(
    title="SSVI fit without the quote screen",
    scene=dict(xaxis_title="Log-moneyness", yaxis_title="Days to expiry", zaxis_title="IV"),
    width=900, height=650,
)
unscreened_fig.show()

## Raw SVI validation diagnostics

The checks below test whether the low in-sample errors generalize, whether calibration depends on its starting values, and whether fitted slices pass basic butterfly-arbitrage conditions. These are diagnostics, not an arbitrage guarantee.

### Held-out strikes

Within each expiry, hold out every fifth strike after sorting by log-moneyness. Fit Raw SVI and an independent single-expiry SSVI-shaped slice to the remaining strikes and compare train/test RMSE in total variance. This comparator is refit using only that expiry; it is not the global shared-parameter SSVI model. Interleaving makes this a test of smile interpolation, not extrapolation.

In [ ]:
holdout_records = []
for expiry, expiry_rows in raw_observations.groupby("expiry_date", sort=True):
    expiry_rows = expiry_rows.sort_values("log_moneyness").reset_index(drop=True)
    test_mask = np.arange(len(expiry_rows)) % 5 == 0
    train_rows = expiry_rows.loc[~test_mask].copy()
    test_rows = expiry_rows.loc[test_mask].copy()

    raw_train = fit_raw_svi_surface(train_rows)
    raw_parameters = raw_train["parameters_by_expiry"].iloc[0]
    raw_test_w = raw_svi_total_variance(
        test_rows["log_moneyness"],
        *(raw_parameters[name] for name in ["a", "b", "rho", "m", "sigma"]),
    )

    single_expiry_ssvi_fit = fit_ssvi_surface(train_rows)
    single_expiry_ssvi_parameters = [single_expiry_ssvi_fit[name] for name in ["rho", "eta", "gamma"]]
    theta = single_expiry_ssvi_fit["theta_by_expiry"]["theta_atm"].iloc[0]
    single_expiry_ssvi_test_w = ssvi_total_variance(
        test_rows["log_moneyness"], theta, *single_expiry_ssvi_parameters
    )
    market_test_w = test_rows["market_w"].to_numpy()
    single_expiry_ssvi_train_rows = single_expiry_ssvi_fit["observations"]

    holdout_records.append({
        "expiry_date": expiry,
        "n_train": len(train_rows),
        "n_test": len(test_rows),
        "raw_train_rmse": raw_parameters["rmse"],
        "raw_test_rmse": np.sqrt(np.mean((raw_test_w - market_test_w) ** 2)),
        "single_expiry_ssvi_train_rmse": np.sqrt(np.mean(single_expiry_ssvi_train_rows["residual_w"] ** 2)),
        "single_expiry_ssvi_test_rmse": np.sqrt(np.mean((single_expiry_ssvi_test_w - market_test_w) ** 2)),
    })

heldout_comparison = pd.DataFrame(holdout_records)
display(heldout_comparison)
display(heldout_comparison[["raw_train_rmse", "raw_test_rmse", "single_expiry_ssvi_train_rmse", "single_expiry_ssvi_test_rmse"]].mean())

### Multi-start calibration

Repeat calibration from three deliberately different parameter starts for expiries nearest 7, 30, 90, and 300 DTE. Compare the fitted curves over each expiry's observed strike range, not only the parameter values: similar curves with different parameters indicate weak parameter identification.

In [ ]:
raw_expiry_table = raw_observations[["expiry_date", "days_to_expiry"]].drop_duplicates()
multistart_records = []
for target in [7, 30, 90, 300]:
    selected = raw_expiry_table.iloc[(raw_expiry_table["days_to_expiry"] - target).abs().argmin()]
    expiry = selected["expiry_date"]
    expiry_rows = raw_observations.loc[raw_observations["expiry_date"].eq(expiry)]
    params_row = raw_result["parameters_by_expiry"].loc[
        raw_result["parameters_by_expiry"]["expiry_date"].eq(expiry)
    ].iloc[0]
    k = expiry_rows["log_moneyness"].to_numpy()
    k_grid = np.linspace(k.min(), k.max(), 300)
    default_params = [params_row[name] for name in ["a", "b", "rho", "m", "sigma"]]
    reference_curve = raw_svi_total_variance(k_grid, *default_params)
    a0 = max(float(expiry_rows["market_w"].min()) - 0.01, -1.0)
    starts = {
        "default": None,
        "negative_skew": [a0, 0.03, -0.8, 0.0, 0.05],
        "symmetric": [a0, 0.15, 0.0, 0.0, 0.2],
        "positive_skew": [a0, 0.3, 0.7, 0.0, 0.35],
    }
    for start_name, start in starts.items():
        try:
            if start_name == "default":
                fitted_params = default_params
                fit_rmse = params_row["rmse"]
            else:
                fit = fit_raw_svi_surface(expiry_rows, initial_guess=start)
                fitted = fit["parameters_by_expiry"].iloc[0]
                fitted_params = [fitted[name] for name in ["a", "b", "rho", "m", "sigma"]]
                fit_rmse = fitted["rmse"]
            fitted_curve = raw_svi_total_variance(k_grid, *fitted_params)
            multistart_records.append({
                "target_dte": target, "expiry_date": expiry, "start": start_name,
                "rmse": fit_rmse,
                "max_curve_difference": np.max(np.abs(fitted_curve - reference_curve)),
                **dict(zip(["a", "b", "rho", "m", "sigma"], fitted_params)),
            })
        except ValueError as error:
            multistart_records.append({"target_dte": target, "expiry_date": expiry, "start": start_name, "error": str(error)})

multistart_comparison = pd.DataFrame(multistart_records)
display(multistart_comparison)

### Basic butterfly-arbitrage checks

For every fitted expiry, check the exact minimum total variance and the Durrleman function

$$g(k)=\left(1-\frac{kw'(k)}{2w(k)}\right)^2-\frac{w'(k)^2}{4}\left(\frac{1}{w(k)}+\frac{1}{4}\right)+\frac{w''(k)}{2},$$

on both the quoted strike range and the wide $[-5,5]$ range. Also check both wing conditions, $b(1+\rho)<2$ and $b(1-\rho)<2$. The wide-grid result distinguishes violations in observed moneyness from those only in extrapolation. Nonnegative $g(k)$ is required across the real line; finite grids remain diagnostics rather than proofs. This checks butterfly arbitrage per slice, not calendar-spread arbitrage across expiries.

In [ ]:
from src.raw_svi import raw_svi_butterfly_diagnostic

arbitrage_records = []
for _, params in raw_result["parameters_by_expiry"].iterrows():
    expiry_rows = raw_observations.loc[raw_observations["expiry_date"].eq(params["expiry_date"])]
    k_min = expiry_rows["log_moneyness"].min()
    k_max = expiry_rows["log_moneyness"].max()
    svi_params = [params[name] for name in ["a", "b", "rho", "m", "sigma"]]
    quoted = raw_svi_butterfly_diagnostic(*svi_params, np.linspace(k_min, k_max, 401))
    wide = raw_svi_butterfly_diagnostic(*svi_params, np.linspace(-5.0, 5.0, 4001))
    outside_grid = np.linspace(-5.0, 5.0, 4001)
    outside_grid = outside_grid[(outside_grid < k_min) | (outside_grid > k_max)]
    extrapolation = raw_svi_butterfly_diagnostic(*svi_params, outside_grid)
    arbitrage_records.append({
        "expiry_date": params["expiry_date"], "quoted_k_min": k_min, "quoted_k_max": k_max,
        "minimum_g_quoted": quoted["minimum_g_on_grid"],
        "minimum_g_wide": wide["minimum_g_on_grid"],
        "minimum_g_extrapolation": extrapolation["minimum_g_on_grid"],
        "g_nonnegative_quoted": quoted["g_nonnegative_on_grid"],
        "g_nonnegative_wide": wide["g_nonnegative_on_grid"],
        "g_nonnegative_extrapolation": extrapolation["g_nonnegative_on_grid"],
        "nonnegative_total_variance": wide["nonnegative_total_variance"],
        "right_wing_condition": wide["right_wing_condition"],
        "left_wing_condition": wide["left_wing_condition"],
    })

arbitrage_checks = pd.DataFrame(arbitrage_records)
arbitrage_checks["passes_sampled_butterfly_checks"] = (
    arbitrage_checks["nonnegative_total_variance"]
    & arbitrage_checks["g_nonnegative_wide"]
    & arbitrage_checks["right_wing_condition"]
    & arbitrage_checks["left_wing_condition"]
)
display(arbitrage_checks)
display(arbitrage_checks[["g_nonnegative_quoted", "g_nonnegative_wide", "g_nonnegative_extrapolation", "right_wing_condition", "left_wing_condition", "passes_sampled_butterfly_checks"]].mean())

constrained_records = []
for expiry, expiry_rows in raw_observations.groupby("expiry_date", sort=True):
    baseline = raw_result["parameters_by_expiry"].set_index("expiry_date").loc[expiry]
    market_w = expiry_rows["market_w"].to_numpy()
    minimum_w = float(market_w.min())
    starts = [
        [minimum_w - b * sigma * np.sqrt(1 - rho**2), b, rho, 0.0, sigma]
        for b, rho, sigma in [(0.01, 0.0, 0.1), (0.03, -0.8, 0.05), (0.15, 0.0, 0.2), (0.3, 0.7, 0.35)]
    ]
    baseline_params = [baseline[name] for name in ["a", "b", "rho", "m", "sigma"]]
    baseline_checks = raw_svi_butterfly_diagnostic(*baseline_params, np.linspace(-5.0, 5.0, 4001))
    baseline_is_feasible = (
        baseline_checks["nonnegative_total_variance"]
        and baseline_checks["minimum_g_on_grid"] >= -1e-8
        and baseline_checks["right_wing_condition"]
        and baseline_checks["left_wing_condition"]
    )
    candidates = [baseline] if baseline_is_feasible else []
    failures = []
    start_guesses = [baseline_params] + starts
    for start in ([] if baseline_is_feasible else start_guesses):
        try:
            candidate = fit_raw_svi_surface(
                expiry_rows, enforce_arbitrage=True, initial_guess=start
            )["parameters_by_expiry"].iloc[0]
            candidates.append(candidate)
        except ValueError as error:
            failures.append(str(error))
    if candidates:
        constrained_params = min(candidates, key=lambda row: row["rmse"])
        constrained_records.append({
            "expiry_date": expiry,
            "status": "baseline feasible" if baseline_is_feasible else f"fit ({len(candidates)}/5 starts)",
            "no_arb_off_rmse": baseline["rmse"],
            "arbitrage_aware_rmse": constrained_params["rmse"],
            "rmse_increase": constrained_params["rmse"] - baseline["rmse"],
        })
    else:
        constrained_records.append({"expiry_date": expiry, "status": failures[0], "no_arb_off_rmse": baseline["rmse"], "arbitrage_aware_rmse": np.nan, "rmse_increase": np.nan})

constrained_comparison = pd.DataFrame(constrained_records)
display(constrained_comparison)
display(constrained_comparison[["no_arb_off_rmse", "arbitrage_aware_rmse", "rmse_increase"]].mean())